Description...

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import yaml

# Styling Plot
crudo.pt.ccortesp_plot_style()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Runs Information

In [2]:
RUNS_INFO =  {  'Ar': [ # July 2026
                        # {'run_number': 16047, 'extra': ''    },
                        # {'run_number': 16057, 'extra': ''    },
                        {'run_number': 16047, 'extra': '20260727'},  # Changes in Irene by Gonzalo (20260727)
                        {'run_number': 16057, 'extra': '20260727'}
                      ],

                'Xe': [

                      ]
             }

In [3]:
SELECTION_CRITERIA = {  'Ar': (lambda x:  (x['nS1'].sum() == 1)
                                        & (x['nS2'].sum() == 1)
                                        & (0      <= x['S1w'].sum() <= 2.5e3 )  # In [ns]
                                      # & (750     <= x['S1e'].sum() <= 1500  ) # In [PE]
                                        & (1000e3 <= x['S1t'].sum()          )  # In [ns]
                                        & (0      <= x['S2w'].sum() <= 100   )  # In [μs]
                                        & (          x['S2t'].sum() <= 1650e3)  # In [ns]
                                ),
                             
                        'Xe': [

                              ]
                      }

# Reconstructed Data

In [4]:
DORO_DICT = {}
SOPH_DICT = {}

for run in RUNS_INFO['Ar']:

    run_id = run['run_number']
    extra  = run['extra']
    # Construct a clean, descriptive dictionary key
    key = f"{run_id}_{extra}" if extra else f"{run_id}"

    print(f"\nRun {key}:")

    # Load and assign the data
    DORO_DICT[key] = crudo.dm.load_run_data(run_id, key='/DST/Events', trigger2=False, extra=extra)
    SOPH_DICT[key] = crudo.dm.load_run_data(run_id, key='/RECO/Events', trigger2=False, extra=extra)


Run 16047_20260727:
/DST/Events: Run 16047 successfully loaded with data shape: (84702, 26)
/RECO/Events: Run 16047 successfully loaded with data shape: (7334031, 12)

Run 16057_20260727:
/DST/Events: Run 16057 successfully loaded with data shape: (130400, 26)
/RECO/Events: Run 16057 successfully loaded with data shape: (5859012, 12)


# Selected Data

In [9]:
%%time
SEL_DICT = {key : {} for key in DORO_DICT.keys()}

for key in DORO_DICT.keys():
    
    print(f"Run {key}:")

    # Dorothea-level
    doro_df = crudo.dm.filter_run_data(DORO_DICT[key], SELECTION_CRITERIA['Ar'])
    doro_evts = doro_df['event'].unique()
    
    # Sophronia-level
    soph_df = SOPH_DICT[key]
    soph_evts = soph_df['event'].unique()

    # Update dataframes
    event_ids = np.intersect1d(doro_evts, soph_evts)
    print(f"You have {len(event_ids)} events in common between Dorothea and Sophronia")
    doro_df, soph_df = crudo.dm.apply_cut_and_update(doro_df, soph_df, event_ids)
    print(f"{soph_df.shape} hits after applying event cut\n")
    SEL_DICT[key]['/DST/Events'] = doro_df
    SEL_DICT[key]['/RECO/Events'] = soph_df

Run 16047_20260727:
Filtered successfully. Data shape: (51647, 26)
You have 51646 events in common between Dorothea and Sophronia
(5436018, 12) hits after applying event cut

Run 16057_20260727:
Filtered successfully. Data shape: (32172, 26)
You have 32172 events in common between Dorothea and Sophronia
(3811045, 12) hits after applying event cut

CPU times: user 43.5 s, sys: 188 ms, total: 43.6 s
Wall time: 43.9 s


# Dataframes Storage

In [10]:
BASE_PATH = '/lhome/ific/c/ccortesp/Data/Cmmssnng/h5/'

for key, dfs in SEL_DICT.items():

    output_filename = os.path.join(BASE_PATH, f"run_{key}_selected.h5")
    print(f"Saving selected data in: {output_filename}")

    with pd.HDFStore(output_filename, mode='w') as store:
        for key, df in dfs.items():
            store.put(key, df, format='table')

Saving selected data in: /lhome/ific/c/ccortesp/Data/Cmmssnng/h5/run_16047_20260727_selected.h5
Saving selected data in: /lhome/ific/c/ccortesp/Data/Cmmssnng/h5/run_16057_20260727_selected.h5
